# Chạy model gốc `VietAI/vit5-large` chưa fine-tune cho bài toán tóm tắt tiếng Việt

Notebook này được tạo từ notebook fine-tune LoRA ban đầu, nhưng đã **loại bỏ toàn bộ phần train/fine-tune** để chạy baseline model gốc.

Mục tiêu chính:

1. Load trực tiếp model gốc `VietAI/vit5-large` từ Hugging Face.
2. Không gắn LoRA, không load adapter, không gọi `trainer.train()`.
3. Generate summary trên `val.parquet` bằng cùng prefix và tham số generate với notebook fine-tune.
4. Tính ROUGE/BLEU, tùy chọn BERTScore.
5. Lưu prediction/metric để so sánh với model đã fine-tune.

> Kết quả notebook này dùng làm **baseline**. Nếu fine-tuned model tốt hơn, các metric của fine-tuned model nên cao hơn baseline trên cùng validation/test set.

## Cell 1 — Cài thư viện

**Nhiệm vụ:** Cài các thư viện cần để load model, đọc dữ liệu, generate summary và tính metric.

**Khác với notebook fine-tune:** Notebook này không cần `peft`, vì không dùng LoRA adapter.

In [1]:
%%capture
# NHIỆM VỤ:
# - Cài thư viện cho inference/evaluation model gốc ViT5-large.
# - %%capture giúp ẩn log pip dài để notebook gọn hơn.

!pip install \
    transformers==4.39.3 \
    accelerate==0.27.2 \
    datasets \
    evaluate \
    rouge-score \
    sacrebleu \
    sentencepiece \
    pyarrow \
    nltk \
    bert-score

# Ý nghĩa từng package:
# transformers==4.39.3:
#   Dùng để load tokenizer/model và generate summary.
# accelerate==0.27.2:
#   Hỗ trợ chạy model trên GPU ổn định hơn.
# datasets:
#   Không bắt buộc cho inference, nhưng giữ lại để tương thích pipeline nếu cần mở rộng.
# evaluate:
#   Load metric ROUGE và SacreBLEU.
# rouge-score:
#   Backend để evaluate tính ROUGE.
# sacrebleu:
#   Backend để evaluate tính BLEU.
# sentencepiece:
#   Tokenizer gốc của T5/ViT5.
# pyarrow:
#   Đọc file .parquet.
# nltk:
#   Thư viện NLP phụ trợ.
# bert-score:
#   Metric đánh giá tương đồng ngữ nghĩa, tùy chọn vì chạy chậm.

## Cell 2 — Import thư viện và khai báo cấu hình toàn cục

**Nhiệm vụ:** Gom toàn bộ cấu hình chính vào một nơi: đường dẫn dữ liệu, tên model, prefix, tham số generate và đường dẫn output.

**Lưu ý:** Để so sánh công bằng với notebook fine-tune, các tham số quan trọng như `TASK_PREFIX`, `MAX_SOURCE_LENGTH`, `MAX_TARGET_LENGTH`, `FINAL_NUM_BEAMS` nên giữ giống notebook fine-tune.

In [2]:
# NHIỆM VỤ:
# - Import thư viện.
# - Khai báo toàn bộ cấu hình cho baseline model gốc.
# - Tạo thư mục output.

import os                 # Làm việc với đường dẫn, tạo thư mục, biến môi trường.
import json               # Lưu metric ra file .json.
import zipfile            # Nén output để tải về.
from pathlib import Path  # Xử lý đường dẫn kiểu object.

import pandas as pd       # Đọc/làm sạch dữ liệu parquet.
import torch              # Framework deep learning chính.
import evaluate           # Load metric ROUGE/BLEU.

from transformers import (
    AutoTokenizer,          # Load tokenizer tương ứng với MODEL_NAME.
    AutoModelForSeq2SeqLM,  # Load model encoder-decoder cho seq2seq.
    set_seed,               # Cố định seed để kết quả dễ tái lập hơn.
)

# Tắt tokenizer parallelism để giảm warning và tránh treo tiến trình.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Cố định seed.
set_seed(42)

# =========================
# 1. Cấu hình model gốc
# =========================

# Tên model gốc trên Hugging Face.
# Đây là model chưa được fine-tune bằng notebook của bạn.
MODEL_NAME = "VietAI/vit5-large"

# Prefix giống notebook fine-tune.
# Việc giữ giống prefix giúp so sánh công bằng hơn.
TASK_PREFIX = "vietnamese summary: "

# =========================
# 2. Cấu hình đường dẫn Colab
# =========================

# Cấu trúc mong muốn:
# /content/NLP/
# ├── data/
# │   ├── train.parquet       # không bắt buộc trong notebook baseline này
# │   ├── val.parquet         # bắt buộc để đánh giá validation
# │   └── test.parquet        # tùy chọn, nếu có test set độc lập
# └── output/
PROJECT_DIR = "/content/NLP"
DATA_DIR = f"{PROJECT_DIR}/data"
OUTPUT_BASE_DIR = f"{PROJECT_DIR}/output"

# Thư mục output riêng cho model gốc.
# Tách riêng để không ghi đè output của model fine-tuned.
OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/vit5-large-base"

# File validation dùng để đánh giá baseline.
VALID_PATH = f"{DATA_DIR}/val.parquet"

# File test độc lập, nếu sau này bạn có.
TEST_PATH = f"{DATA_DIR}/test.parquet"

# Output validation của model gốc.
BASE_PREDICTION_CSV = f"{OUTPUT_DIR}/vit5_large_base_validation_predictions.csv"
BASE_METRICS_JSON = f"{OUTPUT_DIR}/vit5_large_base_validation_metrics.json"
BASE_BERTSCORE_JSON = f"{OUTPUT_DIR}/vit5_large_base_validation_bertscore.json"
BASE_EXAMPLES_CSV = f"{OUTPUT_DIR}/vit5_large_base_examples_for_report.csv"

# Output test của model gốc nếu có test.parquet.
BASE_TEST_PREDICTION_CSV = f"{OUTPUT_DIR}/vit5_large_base_test_predictions.csv"
BASE_TEST_METRICS_JSON = f"{OUTPUT_DIR}/vit5_large_base_test_metrics.json"
BASE_TEST_BERTSCORE_JSON = f"{OUTPUT_DIR}/vit5_large_base_test_bertscore.json"

# File metric của model fine-tuned từ notebook cũ.
# Dùng để so sánh nếu file này đã tồn tại sau khi bạn chạy notebook fine-tune.
FINETUNED_METRICS_JSON = f"{OUTPUT_BASE_DIR}/vit5_large_a100_40gb_validation_metrics.json"
FINETUNED_METRICS_JSON_LEGACY = f"{OUTPUT_BASE_DIR}/vit5_large_validation_metrics.json"

# File zip gom output baseline.
ZIP_PATH = f"{OUTPUT_BASE_DIR}/vit5_large_base_outputs.zip"

# Tạo thư mục nếu chưa tồn tại.
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 3. Cấu hình cột dữ liệu
# =========================

SOURCE_COL = "article"  # Cột chứa văn bản gốc cần tóm tắt.
TARGET_COL = "summary"  # Cột chứa tóm tắt chuẩn/reference.

# =========================
# 4. Cấu hình chạy nhanh/chạy đầy đủ
# =========================

DEBUG_MODE = False
# True:
#   Chỉ generate một phần nhỏ validation để kiểm tra pipeline nhanh.
# False:
#   Generate toàn bộ validation để lấy kết quả baseline chính thức.

DEBUG_VALID_SIZE = 50
# Số mẫu validation khi DEBUG_MODE=True.

VALIDATION_MAX_GENERATE_SAMPLES = None
# None:
#   Nếu DEBUG_MODE=False, generate toàn bộ validation.
#   Nếu DEBUG_MODE=True, generate tối đa DEBUG_VALID_SIZE mẫu.
# Có thể đặt 100 hoặc 200 nếu chỉ muốn chạy thử nhanh.

# =========================
# 5. Cấu hình tokenize/generate
# =========================

MAX_SOURCE_LENGTH = 768
# Số token tối đa của article đầu vào.
# Giữ giống notebook fine-tune để so sánh công bằng.

MAX_TARGET_LENGTH = 160
# Số token tối đa của summary sinh ra.
# Giữ giống notebook fine-tune.

FINAL_GENERATE_BATCH_SIZE = 4
# Batch size khi generate.
# Nếu Colab bị OOM, giảm xuống 2 hoặc 1.

FINAL_NUM_BEAMS = 4
# Số beam search khi generate.
# Giữ giống notebook fine-tune để so sánh công bằng.

FINAL_MIN_LENGTH = 30
# Độ dài tối thiểu của summary sinh ra.

FINAL_NO_REPEAT_NGRAM_SIZE = 3
# Cấm lặp lại n-gram kích thước 3 để giảm lỗi lặp cụm từ.

RUN_BERTSCORE = True
# True để tính BERTScore.
# False để bỏ qua vì chạy chậm và tốn RAM hơn ROUGE/BLEU.

BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"
# Model dùng để tính BERTScore.
# Giữ giống notebook fine-tuned để so sánh công bằng.

METRIC_BATCH_SIZE = 8
# Batch size khi tính BERTScore.
# Nếu bị OOM khi tính BERTScore, giảm xuống 4 hoặc 2.

TOO_SHORT_MIN_TOKENS = 5
# Prediction có ít hơn số token này sẽ bị xem là quá ngắn.

TOO_SHORT_REF_RATIO = 0.5
# Prediction ngắn hơn 50% độ dài reference cũng bị xem là quá ngắn.

# =========================
# 6. Kiểm tra GPU và đường dẫn
# =========================

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Model name:", MODEL_NAME)
print("Validation path:", VALID_PATH)
print("Output directory:", OUTPUT_DIR)
print("Metric config -> BERTScore:", RUN_BERTSCORE, "| model:", BERTSCORE_MODEL_TYPE, "| too_short_min_tokens:", TOO_SHORT_MIN_TOKENS, "| too_short_ref_ratio:", TOO_SHORT_REF_RATIO)


CUDA available: True
GPU: NVIDIA L4
Model name: VietAI/vit5-large
Validation path: /content/NLP/data/val.parquet
Output directory: /content/NLP/output/vit5-large-base
Metric config -> BERTScore: True | model: bert-base-multilingual-cased | too_short_min_tokens: 5 | too_short_ref_ratio: 0.5


## Cell 3 — Kiểm tra file validation

**Nhiệm vụ:** Kiểm tra `val.parquet` có tồn tại đúng đường dẫn không.

**Ý nghĩa:** Baseline này chỉ cần validation để đánh giá model gốc. Không cần `train.parquet` vì không train.

In [3]:
# NHIỆM VỤ:
# - Kiểm tra file validation bắt buộc.

if not os.path.exists(VALID_PATH):
    raise FileNotFoundError(
        "Không tìm thấy file validation. Hãy kiểm tra cấu trúc thư mục Colab:\n"
        f"- Cần có: {VALID_PATH}\n\n"
        "Cấu trúc đúng nên là:\n"
        "/content/NLP/data/val.parquet"
    )

print("Đã tìm thấy file validation:", VALID_PATH)

Đã tìm thấy file validation: /content/NLP/data/val.parquet


## Cell 4 — Đọc và làm sạch dữ liệu validation

**Nhiệm vụ:** Đọc `val.parquet`, kiểm tra cột bắt buộc, xóa dòng rỗng và chuẩn hóa khoảng trắng.

**Ý nghĩa:** Đảm bảo dữ liệu đưa vào model có đúng hai cột sạch: `article` và `summary`.

In [4]:
# NHIỆM VỤ:
# - Đọc dữ liệu parquet.
# - Kiểm tra cột article/summary.
# - Làm sạch text và loại bỏ dòng không hợp lệ.

def clean_text(text):
    """Chuẩn hóa nhẹ văn bản bằng cách ép string, xóa khoảng trắng thừa."""
    return " ".join(str(text).split())


def load_and_clean_parquet(path, source_col=SOURCE_COL, target_col=TARGET_COL):
    """Đọc và làm sạch một file parquet."""
    df = pd.read_parquet(path)

    print(f"\nFile: {path}")
    print("Shape gốc:", df.shape)
    print("Columns:", df.columns.tolist())

    required_cols = [source_col, target_col]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(
                f"Thiếu cột {col!r} trong {path}. "
                f"Các cột hiện có: {df.columns.tolist()}"
            )

    df = df[required_cols].dropna().copy()
    df[source_col] = df[source_col].map(clean_text)
    df[target_col] = df[target_col].map(clean_text)

    df = df[
        (df[source_col].str.len() > 0)
        & (df[target_col].str.len() > 0)
    ].reset_index(drop=True)

    print("Shape sau làm sạch:", df.shape)
    return df


valid_df = load_and_clean_parquet(VALID_PATH)

if DEBUG_MODE:
    valid_df = valid_df.sample(
        n=min(DEBUG_VALID_SIZE, len(valid_df)),
        random_state=42,
    ).reset_index(drop=True)
    print("DEBUG_MODE=True: chỉ dùng validation subset để chạy thử nhanh.")
else:
    valid_df = valid_df.reset_index(drop=True)
    print("DEBUG_MODE=False: dùng toàn bộ validation để đánh giá baseline.")

print("Validation used:", valid_df.shape)
display(valid_df.head(3))


File: /content/NLP/data/val.parquet
Shape gốc: (1349, 2)
Columns: ['article', 'summary']
Shape sau làm sạch: (1349, 2)
DEBUG_MODE=False: dùng toàn bộ validation để đánh giá baseline.
Validation used: (1349, 2)


,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."


## Cell 5 — Load tokenizer và model gốc chưa fine-tune

**Nhiệm vụ:** Tải trực tiếp `VietAI/vit5-large` từ Hugging Face.

**Điểm quan trọng:** Cell này **không gắn LoRA**, **không load adapter**, **không train**. Đây chính là model gốc dùng làm baseline.

In [5]:
# NHIỆM VỤ:
# - Load tokenizer và model gốc ViT5-large.
# - Không dùng LoRA adapter.

# Load tokenizer.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    # use_fast=False:
    #   Dùng SentencePiece tokenizer chuẩn của ViT5.
)

# Chọn dtype phù hợp.
# Nếu có GPU, dùng float16 để giảm VRAM khi inference.
# Nếu chạy CPU, dùng float32 để ổn định hơn.
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load model gốc chưa fine-tune.
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_dtype,
)

# Đưa model lên GPU nếu có.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)
base_model.eval()

print("Tokenizer loaded from:", MODEL_NAME)
print("Tokenizer vocab size:", len(tokenizer))
print("Model vocab size:", base_model.config.vocab_size)
print("Model dtype:", model_dtype)
print("Device:", device)

# Kiểm tra tokenizer và model có cùng vocab size.
assert len(tokenizer) == base_model.config.vocab_size, (
    f"Tokenizer vocab size {len(tokenizer)} != model vocab size {base_model.config.vocab_size}"
)

print("Đã load xong model gốc chưa fine-tune.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


pytorch_model.bin:   0%|          | 0.00/3.17G [00:00<?, ?B/s]

Tokenizer loaded from: VietAI/vit5-large
Tokenizer vocab size: 36100
Model vocab size: 36100
Model dtype: torch.float16
Device: cuda
Đã load xong model gốc chưa fine-tune.


## Cell 6 — Hàm generate summary theo batch

**Nhiệm vụ:** Viết hàm sinh tóm tắt cho danh sách article.

**Ý nghĩa:** Hàm này dùng cùng prefix và tham số generate với notebook fine-tune để kết quả baseline có thể so sánh công bằng.

In [6]:
# NHIỆM VỤ:
# - Tạo hàm generate theo batch để inference nhanh hơn generate từng dòng.

def generate_summaries(texts, batch_size=FINAL_GENERATE_BATCH_SIZE):
    """Sinh summary cho danh sách article bằng model gốc."""

    base_model.eval()
    predictions = []

    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]

        # Thêm prefix nhiệm vụ giống notebook fine-tune.
        batch_inputs = [TASK_PREFIX + clean_text(text) for text in batch_texts]

        # Tokenize batch input.
        inputs = tokenizer(
            batch_inputs,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

        # Đưa tensor lên cùng device với model.
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Inference không cần gradient.
        with torch.no_grad():
            output_ids = base_model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=MAX_TARGET_LENGTH,
                min_length=FINAL_MIN_LENGTH,
                num_beams=FINAL_NUM_BEAMS,
                length_penalty=1.0,
                no_repeat_ngram_size=FINAL_NO_REPEAT_NGRAM_SIZE,
                early_stopping=True,
            )

        # Decode token IDs thành text.
        batch_predictions = tokenizer.batch_decode(
            output_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        predictions.extend([pred.strip() for pred in batch_predictions])

    return predictions

## Cell 7 — Sinh thử vài mẫu validation

**Nhiệm vụ:** Generate thử một vài mẫu để kiểm tra model có chạy được không.

**Lưu ý:** Vì đây là model gốc chưa fine-tune riêng cho dataset của bạn, prediction có thể kém hoặc không giống summary mong muốn. Điều đó là bình thường đối với baseline.

In [7]:
# NHIỆM VỤ:
# - Sinh thử tối đa 3 mẫu validation để kiểm tra pipeline.

NUM_QUICK_EXAMPLES = min(3, len(valid_df))

quick_articles = valid_df[SOURCE_COL].head(NUM_QUICK_EXAMPLES).tolist()
quick_references = valid_df[TARGET_COL].head(NUM_QUICK_EXAMPLES).tolist()
quick_predictions = generate_summaries(
    quick_articles,
    batch_size=min(FINAL_GENERATE_BATCH_SIZE, NUM_QUICK_EXAMPLES),
)

for i, (article, reference, prediction) in enumerate(
    zip(quick_articles, quick_references, quick_predictions),
    start=1,
):
    print("=" * 100)
    print(f"BASE MODEL EXAMPLE {i}")
    print("\nARTICLE:")
    print(article[:800], "...")
    print("\nREFERENCE SUMMARY:")
    print(reference)
    print("\nBASE MODEL PREDICTION:")
    print(prediction)

BASE MODEL EXAMPLE 1

ARTICLE:
Giải thưởng công bố gần đây bởi World Travel Awards. Đây là năm thứ hai liên tiếp InterContinental Phu Quoc Long Beach Resort được vinh danh ở hạng mục gia đình trên toàn châu Á. Khu nghỉ dưỡng tọa lạc bên biển Phú Quốc, nổi bật với thiết kế lấy cảm hứng từ đại dương. Khuôn viên rộng rãi với nhiều mảng xanh thiên nhiên đậm chất nhiệt đới. Không gian sảnh lễ tân, phòng nghỉ, villa, nhà hàng... đều được chú trọng để tạo sự hài hòa với biển và cây cối. Du khách có thể chọn nghỉ ngơi tại khu phòng khách sạn rộng rãi, tiện nghi, hoặc những căn hộ, phòng suite, biệt thự cao cấp hướng biển. Mỗi không gian được thiết kế dựa trên tinh thần gắn kết các thành viên trong gia đình. Bên cạnh tiện nghi sang trọng, InterContinental Phu Quoc còn hút khách gia đình nhờ loạt trải nghiệm giải trí, thư giãn đa ...

REFERENCE SUMMARY:
InterContinental Phu Quoc Long Beach Resort đã được vinh danh là khu nghỉ dưỡng gia đình tốt nhất châu Á lần thứ 2 liên tiếp. Khu nghỉ dưỡng này

## Cell 8 — Generate prediction trên validation

**Nhiệm vụ:** Dùng model gốc sinh summary cho validation set và lưu ra CSV.

**Output:** `vit5_large_base_validation_predictions.csv`.

In [8]:
# NHIỆM VỤ:
# - Generate prediction trên validation.
# - Lưu article, reference, prediction ra CSV.

from tqdm.auto import tqdm

# Quyết định số mẫu validation dùng để generate.
if VALIDATION_MAX_GENERATE_SAMPLES is None:
    max_generate_samples = min(DEBUG_VALID_SIZE, len(valid_df)) if DEBUG_MODE else len(valid_df)
else:
    max_generate_samples = min(VALIDATION_MAX_GENERATE_SAMPLES, len(valid_df))

gen_df = valid_df.iloc[:max_generate_samples].copy().reset_index(drop=True)
print("Validation samples for baseline generation:", len(gen_df))

validation_articles = gen_df[SOURCE_COL].tolist()
validation_references = gen_df[TARGET_COL].tolist()

validation_predictions = []
for start_idx in tqdm(range(0, len(validation_articles), FINAL_GENERATE_BATCH_SIZE)):
    batch_articles = validation_articles[start_idx:start_idx + FINAL_GENERATE_BATCH_SIZE]
    batch_predictions = generate_summaries(
        batch_articles,
        batch_size=FINAL_GENERATE_BATCH_SIZE,
    )
    validation_predictions.extend(batch_predictions)

base_pred_df = pd.DataFrame({
    "article": validation_articles,
    "reference": validation_references,
    "prediction": validation_predictions,
})

base_pred_df.to_csv(BASE_PREDICTION_CSV, index=False, encoding="utf-8-sig")

print("Saved baseline validation predictions to:", BASE_PREDICTION_CSV)
display(base_pred_df.head())

Validation samples for baseline generation: 1349


  0%|          | 0/338 [00:00<?, ?it/s]

Saved baseline validation predictions to: /content/NLP/output/vit5-large-base/vit5_large_base_validation_predictions.csv


,article,reference,prediction
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...,ry+_Z[!]W>>j&WjẲẰÕỸ
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...,)+j[]!>j&WWelcomeỴ Foucault Foucault formosah....
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ...",)+!!!!!_Z[!!>>j&WỴjỠÈẰ]
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ...",+_Z[-----]!>j&WỴẼỠẪẲẺẶỖẸỸ
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...,"+!j[!]!>j&WỴ FoucaultẺẶ khám là,,, là,ỖẸỸÈ"


## Cell 9 — Tính ROUGE, BERTScore, BLEU và Too Short Rate trên validation

In [9]:
# NHIỆM VỤ:
# - Tính các độ đo giống notebook ViT5 fine-tuned A100 40GB.
# - Bao gồm ROUGE, BERTScore, BLEU, Too Short Rate và các metric độ dài.
# - Lưu toàn bộ metric vào JSON dạng phẳng để dễ so sánh với model fine-tuned.

from sacrebleu import corpus_bleu
from bert_score import score as bert_score

# Load ROUGE ở đây để cell này có thể chạy độc lập sau khi đã có base_pred_df.
rouge = evaluate.load("rouge")


def metric_tokenize(text):
    """Tokenize đơn giản cho metric độ dài.

    Dùng split theo khoảng trắng để đo số token ở mức từ/cụm từ đã cách nhau bởi whitespace.
    Hàm này không dùng cho ROUGE/BLEU, chỉ dùng cho length-based metrics.
    """
    return str(text).strip().split()


def compute_rouge_scores(predictions, references):
    """Tính ROUGE bằng evaluate.

    use_stemmer=False vì stemmer của rouge-score chủ yếu dành cho tiếng Anh,
    không phù hợp để stemming tiếng Việt.
    """
    if len(predictions) == 0:
        return {
            "rouge1": 0.0,
            "rouge2": 0.0,
            "rougeL": 0.0,
            "rougeLsum": 0.0,
        }

    result = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False,
    )

    return {key: float(value) for key, value in result.items()}


def compute_bleu_score(predictions, references):
    """Tính corpus BLEU bằng sacreBLEU.

    BLEU không phải metric chính cho summarization, nhưng có ích để tham khảo
    mức trùng n-gram giữa prediction và reference.
    """
    if len(predictions) == 0:
        return {"bleu": 0.0}

    bleu = corpus_bleu(
        predictions,
        [references],
        tokenize="intl",
    )

    return {"bleu": float(bleu.score)}


def compute_bertscore_scores(predictions, references):
    """Tính BERTScore Precision/Recall/F1 trung bình.

    BERTScore bổ sung góc nhìn ngữ nghĩa cho ROUGE/BLEU vì nó so sánh embedding
    thay vì chỉ so trùng token bề mặt.
    """
    if len(predictions) == 0:
        return {
            "bertscore_precision": 0.0,
            "bertscore_recall": 0.0,
            "bertscore_f1": 0.0,
        }

    bert_device = "cuda" if torch.cuda.is_available() else "cpu"

    precision, recall, f1 = bert_score(
        cands=predictions,
        refs=references,
        model_type=BERTSCORE_MODEL_TYPE,
        device=bert_device,
        batch_size=METRIC_BATCH_SIZE,
        verbose=True,
        rescale_with_baseline=False,
    )

    return {
        "bertscore_precision": float(precision.mean().item()),
        "bertscore_recall": float(recall.mean().item()),
        "bertscore_f1": float(f1.mean().item()),
    }


def compute_too_short_rate(predictions, references):
    """Tính tỷ lệ summary sinh ra quá ngắn.

    Một prediction được xem là quá ngắn nếu thỏa ít nhất một trong hai điều kiện:
    1. Số token prediction < TOO_SHORT_MIN_TOKENS.
    2. Số token prediction < TOO_SHORT_REF_RATIO * số token reference.

    Metric này đặc biệt hữu ích để phát hiện model sinh chuỗi rỗng, quá cụt,
    hoặc chỉ sinh vài từ nhưng ROUGE/BLEU không nói rõ nguyên nhân lỗi.
    """
    if len(predictions) == 0:
        return {
            "avg_prediction_tokens": 0.0,
            "avg_reference_tokens": 0.0,
            "avg_length_ratio_pred_ref": 0.0,
            "empty_prediction_rate": 0.0,
            "too_short_rate": 0.0,
            "too_short_min_tokens": TOO_SHORT_MIN_TOKENS,
            "too_short_ref_ratio": TOO_SHORT_REF_RATIO,
        }

    pred_lengths = [len(metric_tokenize(pred)) for pred in predictions]
    ref_lengths = [len(metric_tokenize(ref)) for ref in references]

    empty_flags = [length == 0 for length in pred_lengths]
    too_short_flags = []
    length_ratios = []

    for pred_len, ref_len in zip(pred_lengths, ref_lengths):
        safe_ref_len = max(1, ref_len)
        length_ratio = pred_len / safe_ref_len
        length_ratios.append(length_ratio)

        is_too_short = (
            pred_len < TOO_SHORT_MIN_TOKENS
            or pred_len < TOO_SHORT_REF_RATIO * safe_ref_len
        )
        too_short_flags.append(is_too_short)

    n = len(predictions)
    return {
        "avg_prediction_tokens": float(sum(pred_lengths) / n),
        "avg_reference_tokens": float(sum(ref_lengths) / n),
        "avg_length_ratio_pred_ref": float(sum(length_ratios) / n),
        "empty_prediction_rate": float(sum(empty_flags) / n),
        "too_short_rate": float(sum(too_short_flags) / n),
        "too_short_min_tokens": TOO_SHORT_MIN_TOKENS,
        "too_short_ref_ratio": TOO_SHORT_REF_RATIO,
    }


# Chuẩn bị list prediction/reference từ output của model gốc.
predictions = base_pred_df["prediction"].fillna("").astype(str).tolist()
references = base_pred_df["reference"].fillna("").astype(str).tolist()

# Tính metric.
rouge_metrics = compute_rouge_scores(predictions, references)
bleu_metrics = compute_bleu_score(predictions, references)
length_metrics = compute_too_short_rate(predictions, references)

if RUN_BERTSCORE:
    bertscore_metrics = compute_bertscore_scores(predictions, references)
else:
    bertscore_metrics = {
        "bertscore_precision": None,
        "bertscore_recall": None,
        "bertscore_f1": None,
    }

# Gom metric phẳng để dễ đưa vào bảng so sánh.
base_validation_metrics = {
    "num_validation_samples": len(base_pred_df),
    "num_samples": len(base_pred_df),
    "debug_mode": DEBUG_MODE,
    "model_type": "base_unfinetuned",
    "model_name": MODEL_NAME,
    "fine_tuning_method": None,
    "output_dir": OUTPUT_DIR,
    "task_prefix": TASK_PREFIX,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "final_num_beams": FINAL_NUM_BEAMS,
    "final_min_length": FINAL_MIN_LENGTH,
    "final_no_repeat_ngram_size": FINAL_NO_REPEAT_NGRAM_SIZE,
    "run_bertscore": RUN_BERTSCORE,
    "bert_score_model": BERTSCORE_MODEL_TYPE if RUN_BERTSCORE else None,
    **rouge_metrics,
    **bertscore_metrics,
    **bleu_metrics,
    **length_metrics,
    # Giữ thêm dạng nested để tương thích với phiên bản notebook cũ nếu cần.
    "rouge": rouge_metrics,
}

# In metric dễ đọc.
print("===== BASE MODEL VALIDATION METRICS =====")
for key, value in base_validation_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# Lưu JSON tổng hợp.
with open(BASE_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(base_validation_metrics, f, ensure_ascii=False, indent=2)

print("\nSaved baseline validation metrics to:", BASE_METRICS_JSON)

# Lưu riêng BERTScore để tương thích với các notebook cũ nếu cần.
if RUN_BERTSCORE:
    with open(BASE_BERTSCORE_JSON, "w", encoding="utf-8") as f:
        json.dump(bertscore_metrics, f, ensure_ascii=False, indent=2)
    print("Saved baseline BERTScore to:", BASE_BERTSCORE_JSON)

# Tạo bảng metric để xem nhanh trong notebook.
metric_rows = []
for key, value in base_validation_metrics.items():
    if isinstance(value, (int, float)) and key not in {"debug_mode"}:
        metric_rows.append({"metric": key, "value": value})

base_validation_metric_df = pd.DataFrame(metric_rows)
display(base_validation_metric_df)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  0%|          | 0/338 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/169 [00:00<?, ?it/s]

done in 8.00 seconds, 168.54 sentences/sec
===== BASE MODEL VALIDATION METRICS =====
num_validation_samples: 1349
num_samples: 1349
debug_mode: False
model_type: base_unfinetuned
model_name: VietAI/vit5-large
fine_tuning_method: None
output_dir: /content/NLP/output/vit5-large-base
task_prefix: vietnamese summary: 
max_source_length: 768
max_target_length: 160
final_num_beams: 4
final_min_length: 30
final_no_repeat_ngram_size: 3
run_bertscore: True
bert_score_model: bert-base-multilingual-cased
rouge1: 0.0177
rouge2: 0.0033
rougeL: 0.0166
rougeLsum: 0.0165
bertscore_precision: 0.5458
bertscore_recall: 0.5169
bertscore_f1: 0.5307
bleu: 0.0011
avg_prediction_tokens: 3.0252
avg_reference_tokens: 102.0682
avg_length_ratio_pred_ref: 0.0309
empty_prediction_rate: 0.0000
too_short_rate: 1.0000
too_short_min_tokens: 5
too_short_ref_ratio: 0.5000
rouge: {'rouge1': 0.017673774705523608, 'rouge2': 0.003334658014330601, 'rougeL': 0.016576834562254908, 'rougeLsum': 0.01654705360153139}

Saved baseli

,metric,value
0,num_validation_samples,1349
1,num_samples,1349
2,max_source_length,768
3,max_target_length,160
4,final_num_beams,4
5,final_min_length,30
6,final_no_repeat_ngram_size,3
7,run_bertscore,True
8,rouge1,0.017674
9,rouge2,0.003335


## Cell 10 — Gợi ý đọc các metric validation

In [10]:
# NHIỆM VỤ:
# - In giải thích ngắn về cách đọc metric để dùng trong báo cáo.
# - Cell này không tính toán thêm, chỉ giải thích các độ đo đã tính ở Cell 9.

print("""
Cách đọc metric validation:

1. ROUGE-1:
   - Đo mức trùng từ đơn giữa prediction và reference.
   - Cao hơn thường nghĩa là model giữ được nhiều từ khóa/ý chính hơn.

2. ROUGE-2:
   - Đo mức trùng cụm 2 từ liên tiếp.
   - Đây là chỉ số khó hơn ROUGE-1; tăng ROUGE-2 thường cho thấy câu tóm tắt bám diễn đạt tốt hơn.

3. ROUGE-L / ROUGE-Lsum:
   - Đo chuỗi con chung dài nhất, quan tâm đến thứ tự thông tin.
   - Phù hợp để đánh giá cấu trúc summary.

4. BERTScore:
   - So sánh gần nghĩa bằng embedding.
   - Hữu ích khi prediction diễn đạt khác reference nhưng vẫn cùng ý.

5. BLEU:
   - Đo trùng n-gram, thường dùng nhiều trong dịch máy.
   - Với summarization, nên xem là metric phụ.

6. Too Short Rate:
   - Tỷ lệ prediction quá ngắn.
   - Nếu chỉ số này cao, model có dấu hiệu sinh summary cụt/rỗng dù một số metric khác có thể không phản ánh rõ.

7. Empty Prediction Rate:
   - Tỷ lệ prediction rỗng.
   - Nếu chỉ số này cao, pipeline generate hoặc model có vấn đề nghiêm trọng.

8. Average Length Ratio:
   - Tỷ lệ độ dài trung bình prediction/reference.
   - Quá thấp: summary có thể bị cụt. Quá cao: summary có thể quá dài hoặc copy nhiều.
""")



Cách đọc metric validation:

1. ROUGE-1:
   - Đo mức trùng từ đơn giữa prediction và reference.
   - Cao hơn thường nghĩa là model giữ được nhiều từ khóa/ý chính hơn.

2. ROUGE-2:
   - Đo mức trùng cụm 2 từ liên tiếp.
   - Đây là chỉ số khó hơn ROUGE-1; tăng ROUGE-2 thường cho thấy câu tóm tắt bám diễn đạt tốt hơn.

3. ROUGE-L / ROUGE-Lsum:
   - Đo chuỗi con chung dài nhất, quan tâm đến thứ tự thông tin.
   - Phù hợp để đánh giá cấu trúc summary.

4. BERTScore:
   - So sánh gần nghĩa bằng embedding.
   - Hữu ích khi prediction diễn đạt khác reference nhưng vẫn cùng ý.

5. BLEU:
   - Đo trùng n-gram, thường dùng nhiều trong dịch máy.
   - Với summarization, nên xem là metric phụ.

6. Too Short Rate:
   - Tỷ lệ prediction quá ngắn.
   - Nếu chỉ số này cao, model có dấu hiệu sinh summary cụt/rỗng dù một số metric khác có thể không phản ánh rõ.

7. Empty Prediction Rate:
   - Tỷ lệ prediction rỗng.
   - Nếu chỉ số này cao, pipeline generate hoặc model có vấn đề nghiêm trọng.

8. Average L

## Cell 11 — Lưu ví dụ định tính cho báo cáo

**Nhiệm vụ:** Lấy ngẫu nhiên một số prediction của model gốc để so sánh với model fine-tuned.

**Output:** `vit5_large_base_examples_for_report.csv`.

In [11]:
# NHIỆM VỤ:
# - Lấy ví dụ định tính từ base_pred_df.
# - Lưu ra CSV để đưa vào báo cáo.

NUM_EXAMPLES_FOR_REPORT = min(10, len(base_pred_df))

base_examples_df = base_pred_df.sample(
    n=NUM_EXAMPLES_FOR_REPORT,
    random_state=42,
).copy()

base_examples_df.to_csv(BASE_EXAMPLES_CSV, index=False, encoding="utf-8-sig")

print("Saved baseline examples to:", BASE_EXAMPLES_CSV)
display(base_examples_df)

Saved baseline examples to: /content/NLP/output/vit5-large-base/vit5_large_base_examples_for_report.csv


,article,reference,prediction
289,"TheoThe Paper, khán giả vốn quen Giả Linh với ...","Diễn viên Giả Linh, nổi tiếng với hình tượng m...",+jY[WỴẼỠẪẲẰÕẺẶoloỖlum (lum)lumộcẸlum()ỸlumlumÈỮ
1036,"Phút 43, trận đấu trên sân Sultan Ibrahim (ban...",Trong trận đấu giữa Buriram United (Thái Lan) ...,)+! Thej[!]!!!!>jkampkampkamp&kampkampWkampkam...
535,Ngân hàng Trung ương Nga cuối tuần trước thông...,Ngân hàng Trung ương Nga vừa thông báo giá trị...,) ().+_Z[ (). (). (-Ẳ WẰÕ
346,"Theo đại diện tuyển sinh các trường, đây là th...",Ngày 17/8 là thời điểm kết thúc việc xử lý ngu...,")+!_)Z[!!!!]È,]!"
1075,"""Có nhiều thứ của Liverpool khiến tôi nhớ đến ...",Gary Neville ca ngợi sự đa dạng chiến thuật củ...,+ Foucaultiganigan_Z[!!!!!!!]!!!
782,Ngoài chính sách giảm 50% lệ phí trước bạ từ C...,Chính phủ đang áp dụng chính sách giảm 50% lệ ...,"rize'+_Z[,,!,++.!+_!Ẽ"
303,Con trai họa sĩ - ông Hồ Hồng Lĩnh - cho biết ...,Họa sĩ Hồ Hữu Thủ sinh năm 1940 tại Thủ Dầu Mộ...,")+!!!!!!_Z[!!!>]È""[Ự, 1"
754,Mẫu concept EV Fun chưa phải là phiên bản sản ...,Honda đã trình diễn hai mẫu xe điện tại triển ...,"CC+_Z[!,!!.!_, điện và cơẲ+"
536,"Chốt phiên giao dịch 11/11, giá vàng thế giới ...","Sau khi ông Donald Trump đắc cử Tổng thống Mỹ,...",)+!!_Z[!..]__]!].]
76,Khoảng 200 con khỉ hôm 16/11 lang thang trên đ...,Khoảng 200 con khỉ lang thang trên đường phố v...,"+ đã vàj[!!]!>j, Bangkok, Thái, Bangkok, Thái ..."


## Cell 12 — So sánh nhanh metric model gốc và model fine-tuned

**Nhiệm vụ:** Nếu đã chạy notebook fine-tune trước đó và file metric còn tồn tại, cell này sẽ đọc metric của hai model rồi hiển thị cạnh nhau.

**Lưu ý:** Chỉ nên kết luận nếu cả hai model được generate trên cùng validation set, cùng prefix và cùng tham số generate.

In [12]:
# NHIỆM VỤ:
# - Đọc metric baseline và metric fine-tuned nếu có.
# - Tạo bảng so sánh nhanh.
# - Hỗ trợ cả format metric cũ dạng nested và format mới dạng phẳng.

def get_metric_value(metrics, key):
    """Lấy metric từ JSON dạng phẳng hoặc dạng nested cũ."""
    if key in metrics:
        return metrics.get(key)

    if key.startswith("rouge"):
        return metrics.get("rouge", {}).get(key)

    return None


def flatten_metrics(metrics, model_label):
    """Chuyển JSON metric thành một dòng bảng."""
    return {
        "model": model_label,
        "num_samples": (
            metrics.get("num_validation_samples")
            or metrics.get("num_samples")
        ),
        "rouge1": get_metric_value(metrics, "rouge1"),
        "rouge2": get_metric_value(metrics, "rouge2"),
        "rougeL": get_metric_value(metrics, "rougeL"),
        "rougeLsum": get_metric_value(metrics, "rougeLsum"),
        "bertscore_precision": get_metric_value(metrics, "bertscore_precision"),
        "bertscore_recall": get_metric_value(metrics, "bertscore_recall"),
        "bertscore_f1": get_metric_value(metrics, "bertscore_f1"),
        "bleu": get_metric_value(metrics, "bleu"),
        "too_short_rate": get_metric_value(metrics, "too_short_rate"),
        "empty_prediction_rate": get_metric_value(metrics, "empty_prediction_rate"),
        "avg_prediction_tokens": get_metric_value(metrics, "avg_prediction_tokens"),
        "avg_reference_tokens": get_metric_value(metrics, "avg_reference_tokens"),
        "avg_length_ratio_pred_ref": get_metric_value(metrics, "avg_length_ratio_pred_ref"),
    }


def first_existing_path(paths):
    """Trả về đường dẫn đầu tiên tồn tại trong danh sách."""
    for path in paths:
        if path and os.path.exists(path):
            return path
    return None


rows = []

if os.path.exists(BASE_METRICS_JSON):
    with open(BASE_METRICS_JSON, "r", encoding="utf-8") as f:
        base_metrics_loaded = json.load(f)
    rows.append(flatten_metrics(base_metrics_loaded, "VietAI/vit5-large gốc"))
else:
    print("Chưa có metric baseline:", BASE_METRICS_JSON)

finetuned_metric_path = first_existing_path([
    FINETUNED_METRICS_JSON,
    FINETUNED_METRICS_JSON_LEGACY,
])

if finetuned_metric_path is not None:
    with open(finetuned_metric_path, "r", encoding="utf-8") as f:
        finetuned_metrics_loaded = json.load(f)
    print("Đang dùng metric fine-tuned từ:", finetuned_metric_path)
    rows.append(flatten_metrics(finetuned_metrics_loaded, "VietAI/vit5-large + LoRA fine-tuned"))
else:
    print("Chưa tìm thấy metric fine-tuned.")
    print("Đường dẫn đang kiểm tra:")
    print("-", FINETUNED_METRICS_JSON)
    print("-", FINETUNED_METRICS_JSON_LEGACY)
    print("Hãy chạy notebook fine-tune trước nếu muốn so sánh tự động.")

if rows:
    comparison_df = pd.DataFrame(rows)
    display(comparison_df)

    if len(rows) == 2:
        delta = comparison_df.iloc[1].copy()
        numeric_cols = [
            "rouge1",
            "rouge2",
            "rougeL",
            "rougeLsum",
            "bertscore_precision",
            "bertscore_recall",
            "bertscore_f1",
            "bleu",
            "too_short_rate",
            "empty_prediction_rate",
            "avg_prediction_tokens",
            "avg_reference_tokens",
            "avg_length_ratio_pred_ref",
        ]

        for col in numeric_cols:
            if col in comparison_df.columns:
                base_value = comparison_df.iloc[0][col]
                tuned_value = comparison_df.iloc[1][col]

                if pd.notna(base_value) and pd.notna(tuned_value):
                    delta[col] = tuned_value - base_value
                else:
                    delta[col] = None

        delta["model"] = "Chênh lệch fine-tuned - gốc"
        delta["num_samples"] = comparison_df.iloc[1]["num_samples"]
        display(pd.DataFrame([delta]))


Đang dùng metric fine-tuned từ: /content/NLP/output/vit5_large_a100_40gb_validation_metrics.json


,model,num_samples,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,bleu,too_short_rate,empty_prediction_rate,avg_prediction_tokens,avg_reference_tokens,avg_length_ratio_pred_ref
0,VietAI/vit5-large gốc,1349,0.017674,0.003335,0.016577,0.016547,0.545762,0.516942,0.530723,0.001079,1.000000,0.0,3.025204,102.068199,0.030925
1,VietAI/vit5-large + LoRA fine-tuned,1349,0.744159,0.485589,0.503365,0.503129,0.795353,0.801514,0.798111,35.243103,0.003706,0.0,102.093403,102.068199,1.039769


,model,num_samples,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1,bleu,too_short_rate,empty_prediction_rate,avg_prediction_tokens,avg_reference_tokens,avg_length_ratio_pred_ref
1,Chênh lệch fine-tuned - gốc,1349,0.726485,0.482254,0.486788,0.486582,0.24959,0.284571,0.267388,35.242024,-0.996294,0.0,99.068199,0.0,1.008844


## Cell 13 — Tùy chọn đánh giá trên test set độc lập

**Nhiệm vụ:** Nếu bạn có `/content/NLP/data/test.parquet`, cell này sẽ generate và tính metric trên test set.

**Ý nghĩa:** Test set độc lập giúp đánh giá khách quan hơn validation set.

In [13]:
# # NHIỆM VỤ:
# # - Nếu có test.parquet, chạy baseline trên test set.
# # - Nếu chưa có test.parquet, bỏ qua an toàn.
# # - Metric test dùng cùng bộ độ đo với validation: ROUGE, BERTScore, BLEU, Too Short Rate.

# if os.path.exists(TEST_PATH):
#     print("Tìm thấy test set:", TEST_PATH)
#     test_df = load_and_clean_parquet(TEST_PATH)

#     TEST_MAX_SAMPLES = None
#     # None nghĩa là đánh giá toàn bộ test set.
#     # Có thể đặt 100 hoặc 200 để chạy thử nhanh.

#     if TEST_MAX_SAMPLES is not None:
#         test_df = test_df.iloc[:min(TEST_MAX_SAMPLES, len(test_df))].copy().reset_index(drop=True)

#     test_articles = test_df[SOURCE_COL].tolist()
#     test_references = test_df[TARGET_COL].tolist()

#     test_predictions = []
#     for start_idx in tqdm(range(0, len(test_articles), FINAL_GENERATE_BATCH_SIZE)):
#         batch_articles = test_articles[start_idx:start_idx + FINAL_GENERATE_BATCH_SIZE]
#         batch_predictions = generate_summaries(
#             batch_articles,
#             batch_size=FINAL_GENERATE_BATCH_SIZE,
#         )
#         test_predictions.extend(batch_predictions)

#     base_test_pred_df = pd.DataFrame({
#         "article": test_articles,
#         "reference": test_references,
#         "prediction": test_predictions,
#     })

#     base_test_pred_df.to_csv(BASE_TEST_PREDICTION_CSV, index=False, encoding="utf-8-sig")

#     test_predictions_list = base_test_pred_df["prediction"].fillna("").astype(str).tolist()
#     test_references_list = base_test_pred_df["reference"].fillna("").astype(str).tolist()

#     test_rouge_metrics = compute_rouge_scores(test_predictions_list, test_references_list)
#     test_bleu_metrics = compute_bleu_score(test_predictions_list, test_references_list)
#     test_length_metrics = compute_too_short_rate(test_predictions_list, test_references_list)

#     if RUN_BERTSCORE:
#         test_bertscore_metrics = compute_bertscore_scores(test_predictions_list, test_references_list)
#     else:
#         test_bertscore_metrics = {
#             "bertscore_precision": None,
#             "bertscore_recall": None,
#             "bertscore_f1": None,
#         }

#     base_test_metrics = {
#         "num_test_samples": len(base_test_pred_df),
#         "num_samples": len(base_test_pred_df),
#         "model_type": "base_unfinetuned",
#         "model_name": MODEL_NAME,
#         "fine_tuning_method": None,
#         "task_prefix": TASK_PREFIX,
#         "max_source_length": MAX_SOURCE_LENGTH,
#         "max_target_length": MAX_TARGET_LENGTH,
#         "final_num_beams": FINAL_NUM_BEAMS,
#         "final_min_length": FINAL_MIN_LENGTH,
#         "final_no_repeat_ngram_size": FINAL_NO_REPEAT_NGRAM_SIZE,
#         "run_bertscore": RUN_BERTSCORE,
#         "bert_score_model": BERTSCORE_MODEL_TYPE if RUN_BERTSCORE else None,
#         **test_rouge_metrics,
#         **test_bertscore_metrics,
#         **test_bleu_metrics,
#         **test_length_metrics,
#         "rouge": test_rouge_metrics,
#     }

#     with open(BASE_TEST_METRICS_JSON, "w", encoding="utf-8") as f:
#         json.dump(base_test_metrics, f, ensure_ascii=False, indent=2)

#     if RUN_BERTSCORE:
#         with open(BASE_TEST_BERTSCORE_JSON, "w", encoding="utf-8") as f:
#             json.dump(test_bertscore_metrics, f, ensure_ascii=False, indent=2)

#     print("===== BASE MODEL TEST METRICS =====")
#     for key, value in base_test_metrics.items():
#         if isinstance(value, float):
#             print(f"{key}: {value:.4f}")
#         else:
#             print(f"{key}: {value}")

#     print("Saved test predictions to:", BASE_TEST_PREDICTION_CSV)
#     print("Saved test metrics to:", BASE_TEST_METRICS_JSON)
#     display(base_test_pred_df.head())
# else:
#     print("Chưa có test.parquet, bỏ qua test evaluation.")
#     print("Đường dẫn đang kiểm tra:", TEST_PATH)


## Cell 14 — Demo với văn bản tự nhập

**Nhiệm vụ:** Nhập một article bất kỳ và xem model gốc tóm tắt như thế nào.

**Ý nghĩa:** Cell này chỉ để demo nhanh, không dùng làm metric chính.

In [14]:
# NHIỆM VỤ:
# - Chạy thử model gốc với một văn bản tự nhập.

# @title Chạy thử model gốc với văn bản của bạn

custom_article = "Redmi Note 12 Turbo là một trong những mẫu smartphone tầm trung nổi bật nhất của Xiaomi khi ra mắt vào tháng 3 năm 2023. Đây cũng là chiếc điện thoại đầu tiên trên thế giới được trang bị vi xử lý Snapdragon 7+ Gen 2, con chip mang lại hiệu năng mạnh mẽ và tiệm cận với nhiều thiết bị cao cấp trên thị trường. Nhờ được sản xuất trên tiến trình 4nm của TSMC, bộ xử lý này không chỉ cho khả năng vận hành mượt mà mà còn tối ưu điện năng hiệu quả. Người dùng có thể dễ dàng thực hiện các tác vụ hằng ngày như lướt web, xem phim, làm việc trực tuyến hay chơi những tựa game đồ họa nặng như PUBG Mobile, Call of Duty Mobile hoặc Genshin Impact với mức thiết lập cao mà vẫn duy trì được độ ổn định tốt. Bên cạnh hiệu năng mạnh mẽ, Redmi Note 12 Turbo còn được trang bị màn hình OLED kích thước 6,67 inch với độ phân giải Full HD+, tần số quét 120Hz cùng công nghệ HDR10+ và Dolby Vision. Nhờ đó, thiết bị mang đến chất lượng hiển thị sắc nét, màu sắc sống động và trải nghiệm vuốt chạm mượt mà trong mọi tác vụ. Tuy nhiên, dù sở hữu nhiều ưu điểm nổi bật, Redmi Note 12 Turbo không còn là lựa chọn thực sự hấp dẫn trong năm 2025 do không được phân phối chính hãng tại Việt Nam, gây hạn chế về bảo hành và hỗ trợ kỹ thuật." # @param {type:"string"}

print("Đang tạo tóm tắt bằng model gốc...")

custom_prediction = generate_summaries(
    [custom_article],
    batch_size=1,
)[0]

print("\n" + "=" * 80)
print("📝 BÀI VIẾT GỐC:")
print(custom_article)
print("-" * 80)
print("✨ TÓM TẮT CỦA MODEL GỐC:")
print(custom_prediction)
print("=" * 80)

Đang tạo tóm tắt bằng model gốc...

📝 BÀI VIẾT GỐC:
Redmi Note 12 Turbo là một trong những mẫu smartphone tầm trung nổi bật nhất của Xiaomi khi ra mắt vào tháng 3 năm 2023. Đây cũng là chiếc điện thoại đầu tiên trên thế giới được trang bị vi xử lý Snapdragon 7+ Gen 2, con chip mang lại hiệu năng mạnh mẽ và tiệm cận với nhiều thiết bị cao cấp trên thị trường. Nhờ được sản xuất trên tiến trình 4nm của TSMC, bộ xử lý này không chỉ cho khả năng vận hành mượt mà mà còn tối ưu điện năng hiệu quả. Người dùng có thể dễ dàng thực hiện các tác vụ hằng ngày như lướt web, xem phim, làm việc trực tuyến hay chơi những tựa game đồ họa nặng như PUBG Mobile, Call of Duty Mobile hoặc Genshin Impact với mức thiết lập cao mà vẫn duy trì được độ ổn định tốt. Bên cạnh hiệu năng mạnh mẽ, Redmi Note 12 Turbo còn được trang bị màn hình OLED kích thước 6,67 inch với độ phân giải Full HD+, tần số quét 120Hz cùng công nghệ HDR10+ và Dolby Vision. Nhờ đó, thiết bị mang đến chất lượng hiển thị sắc nét, màu sắc sống

## Cell 15 — Nén output baseline để tải về

**Nhiệm vụ:** Gom prediction CSV, metric JSON, ví dụ báo cáo và test output nếu có vào một file `.zip`.

**Output:** `/content/NLP/output/vit5_large_base_outputs.zip`.

In [15]:
# NHIỆM VỤ:
# - Nén các file output baseline quan trọng.

output_base = Path(OUTPUT_BASE_DIR)
zip_path = Path(ZIP_PATH)

items_to_zip = [
    Path(OUTPUT_DIR),
]

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for item in items_to_zip:
        if not item.exists():
            continue

        if item.is_dir():
            for file_path in item.rglob("*"):
                if file_path.is_file() and file_path != zip_path:
                    zf.write(
                        file_path,
                        arcname=file_path.relative_to(output_base),
                    )
        else:
            zf.write(
                item,
                arcname=item.relative_to(output_base),
            )

print("Output zip:", zip_path)

Output zip: /content/NLP/output/vit5_large_base_outputs.zip
